In [15]:
!pip install pyspark

In [16]:
# Initialize Spark session and import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.appName("TransactionDataAnalysis").getOrCreate()

In [17]:
# Load the transaction data from a CSV file
transaction_df = spark.read.csv("data.csv",header=True,inferSchema=True)

# Display the first 10 rows of the DataFrame
transaction_df.limit(10).toPandas()

,user_id,transaction_date,region,product_category,sale_amount,status,city,age,subscription,raw_timestamp,email,username,price,store_id
0,USR_1007,2026-06-13,West,Home,177.20,Active,Los Angeles,26,Standard,2026-06-11 07:02:36,user_1@example.com,user_1,204.76,STORE_5
1,USR_1007,2026-06-13,West,Home,177.20,Active,Los Angeles,26,Standard,2026-06-11 07:02:36,user_1@example.com,user_1,204.76,STORE_5
2,USR_1001,2026-06-15,West,Home,412.94,None,Los Angeles,46,Premium,2026-06-12 03:12:01,user_2@example.com,user_2,NaN,STORE_4
3,USR_1047,2026-06-09,West,Clothing,480.79,Active,Los Angeles,59,Standard,2026-06-08 10:26:09,user_3@example.com,user_3,185.04,STORE_5
4,USR_1017,2026-06-15,East,Books,416.07,Completed,Los Angeles,46,Basic,2026-06-07 22:13:26,user_4@example.com,user_4,108.60,STORE_1
5,USR_1015,2026-06-01,West,Beauty,223.39,Active,Los Angeles,38,Basic,2026-06-09 19:58:30,user_5@example.com,user_5,225.74,STORE_2
6,USR_1014,2026-06-10,West,Beauty,483.08,None,Los Angeles,48,Basic,2026-06-14 23:46:03,user_6@example.com,user_6,186.74,STORE_5
7,USR_1008,2026-06-06,West,Home,406.02,Completed,Los Angeles,32,Basic,2026-06-12 04:33:13,user_7@example.com,user_7,69.09,STORE_3
8,USR_1047,2026-06-08,East,Books,246.91,Completed,Los Angeles,20,Standard,2026-06-09 10:42:30,user_8@example.com,user_8,46.30,STORE_5
9,USR_1006,2026-06-01,South,Books,368.20,None,Los Angeles,42,Standard,2026-06-09 12:20:11,None,user_9,220.35,STORE_2


In [18]:
# Print the schema and column names
transaction_df.printSchema()
print(transaction_df.columns)

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)

['user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'status', 'city', 'age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'store_id']


In [19]:
# Data Cleaning: Identify duplicate records
num_duplicates = transaction_df.count() - transaction_df.dropDuplicates(["user_id","transaction_date"]).count()
print(f"Number of duplicate records found: {num_duplicates}")

Number of duplicate records found: 36


In [20]:
# Remove duplicate records based on user_id and transaction_date
unique_transactions_df = transaction_df.dropDuplicates(["user_id","transaction_date"])
print(f"Number of unique transaction records after dropping duplicates: {unique_transactions_df.count()}")

Number of unique transaction records after dropping duplicates: 164


In [21]:
transaction_df.count()
# Drop rows containing any null values
df_no_nulls = transaction_df.na.drop()
print(f"Number of rows after dropping all nulls: {df_no_nulls.count()}")

Number of rows after dropping all nulls: 120


In [22]:
# Handling missing 'status' values
null_status_count = transaction_df.filter(col("status").isNull()).count()
print(f"Number of records with null 'status': {null_status_count}")

Number of records with null 'status': 45


In [23]:
transaction_df_filled_status = transaction_df.na.fill(value="Unknown", subset=["status"])

In [24]:
new_null_status_count = transaction_df_filled_status.filter(col("status").isNull()).count()
print(f"Number of records with null 'status' after filling: {new_null_status_count}")

Number of records with null 'status' after filling: 0


In [25]:
transaction_df_filled_status.select("user_id", "status").show(10)

+--------+---------+
| user_id|   status|
+--------+---------+
|USR_1007|   Active|
|USR_1007|   Active|
|USR_1001|  Unknown|
|USR_1047|   Active|
|USR_1017|Completed|
|USR_1015|   Active|
|USR_1014|  Unknown|
|USR_1008|Completed|
|USR_1047|Completed|
|USR_1006|  Unknown|
+--------+---------+
only showing top 10 rows


In [26]:
# Check for inconsistent 'age' data (e.g., age outside a reasonable range)
invalid_age_records = transaction_df.filter((col("age") < 0) | (col("age") > 100))
print("Records with invalid age values:")
invalid_age_records.toPandas()

Records with invalid age values:


,user_id,transaction_date,region,product_category,sale_amount,status,city,age,subscription,raw_timestamp,email,username,price,store_id


In [27]:
# Filtering Data: By Age and Subscription Type

# Filter by age between 18 and 30 with 'Premium' subscription
from pyspark.sql.functions import col
premium_young_adults_df = transaction_df.filter((col("age") >= 18) & (col("age") <= 30) & (col("subscription") == "Premium"))
premium_young_adults_df.toPandas()

,user_id,transaction_date,region,product_category,sale_amount,status,city,age,subscription,raw_timestamp,email,username,price,store_id
0,USR_1005,2026-06-01,South,Beauty,145.37,None,Los Angeles,20,Premium,2026-06-06 17:59:59,user_18@example.com,user_18,120.57,STORE_3
1,USR_1041,2026-06-02,North,Electronics,270.15,Active,Los Angeles,25,Premium,2026-06-09 22:17:08,user_27@example.com,user_27,143.95,STORE_1
2,USR_1027,2026-06-14,West,Clothing,NaN,None,Los Angeles,19,Premium,2026-06-09 23:46:54,None,user_39,101.39,STORE_4
3,USR_1006,2026-06-12,West,Beauty,35.17,Pending,Los Angeles,25,Premium,2026-06-02 10:18:29,user_46@example.com,user_46,NaN,STORE_2
4,USR_1016,2026-06-11,North,Clothing,307.10,Completed,Los Angeles,23,Premium,2026-06-13 09:36:43,user_53@example.com,user_53,262.67,STORE_3
5,USR_1046,2026-06-08,West,Beauty,230.75,Active,Los Angeles,30,Premium,2026-06-14 20:57:25,user_55@example.com,user_55,103.83,STORE_1
6,USR_1040,2026-06-06,West,Beauty,NaN,Pending,Los Angeles,24,Premium,2026-06-11 11:44:15,user_63@example.com,user_63,223.53,STORE_1
7,USR_1004,2026-06-04,East,Clothing,333.98,Completed,Los Angeles,19,Premium,2026-06-04 04:30:09,user_69@example.com,user_69,NaN,STORE_4
8,USR_1024,2026-06-01,South,Clothing,251.96,Active,Los Angeles,30,Premium,2026-06-09 16:12:36,None,user_78,263.71,STORE_2
9,USR_1017,2026-06-14,North,Home,29.53,Pending,Los Angeles,22,Premium,2026-06-09 04:10:20,user_79@example.com,user_79,233.19,STORE_4


In [28]:
# Filter by region: 'West'
west_region_df = transaction_df.filter(col("region") == "West")
west_region_df.toPandas()

,user_id,transaction_date,region,product_category,sale_amount,status,city,age,subscription,raw_timestamp,email,username,price,store_id
0,USR_1007,2026-06-13,West,Home,177.20,Active,Los Angeles,26,Standard,2026-06-11 07:02:36,user_1@example.com,user_1,204.76,STORE_5
1,USR_1007,2026-06-13,West,Home,177.20,Active,Los Angeles,26,Standard,2026-06-11 07:02:36,user_1@example.com,user_1,204.76,STORE_5
2,USR_1001,2026-06-15,West,Home,412.94,None,Los Angeles,46,Premium,2026-06-12 03:12:01,user_2@example.com,user_2,NaN,STORE_4
3,USR_1047,2026-06-09,West,Clothing,480.79,Active,Los Angeles,59,Standard,2026-06-08 10:26:09,user_3@example.com,user_3,185.04,STORE_5
4,USR_1015,2026-06-01,West,Beauty,223.39,Active,Los Angeles,38,Basic,2026-06-09 19:58:30,user_5@example.com,user_5,225.74,STORE_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,USR_1046,2026-06-05,West,Electronics,333.00,Pending,San Francisco,50,Basic,2026-06-05 21:52:54,user_183@example.com,user_183,261.12,STORE_3
80,USR_1006,2026-06-01,West,Electronics,96.06,None,Los Angeles,49,Standard,2026-06-07 20:50:21,user_189@example.com,user_189,200.33,STORE_5
81,USR_1012,2026-06-08,West,Electronics,175.97,Active,New York,52,Basic,2026-06-15 17:36:42,user_195@example.com,user_195,267.43,STORE_2
82,USR_1009,2026-06-09,West,Beauty,481.82,Active,Chicago,24,Standard,2026-06-01 05:59:54,user_196@example.com,user_196,110.26,STORE_5


In [29]:
# Filter by product category: 'Electronics'
electronics_df = transaction_df.filter(col("product_category") == "Electronics")
electronics_df.toPandas()

,user_id,transaction_date,region,product_category,sale_amount,status,city,age,subscription,raw_timestamp,email,username,price,store_id
0,USR_1027,2026-06-13,West,Electronics,99.10,Completed,Los Angeles,33,Standard,2026-06-05 07:19:49,user_15@example.com,user_15,NaN,STORE_1
1,USR_1045,2026-06-14,East,Electronics,294.96,None,Los Angeles,42,Standard,2026-06-14 17:17:59,user_26@example.com,user_26,166.18,STORE_2
2,USR_1041,2026-06-02,North,Electronics,270.15,Active,Los Angeles,25,Premium,2026-06-09 22:17:08,user_27@example.com,user_27,143.95,STORE_1
3,USR_1014,2026-06-03,South,Electronics,104.37,None,Los Angeles,43,Standard,2026-06-04 01:06:26,user_31@example.com,user_31,122.40,STORE_1
4,USR_1014,2026-06-03,South,Electronics,104.37,None,Los Angeles,43,Standard,2026-06-04 01:06:26,user_31@example.com,user_31,122.40,STORE_1
5,USR_1017,2026-06-07,East,Electronics,281.15,Active,Los Angeles,55,Basic,2026-06-03 13:56:53,user_41@example.com,user_41,194.73,STORE_5
6,USR_1017,2026-06-07,East,Electronics,281.15,Active,Los Angeles,55,Basic,2026-06-03 13:56:53,user_41@example.com,user_41,194.73,STORE_5
7,USR_1009,2026-06-04,West,Electronics,71.88,Completed,Los Angeles,40,Premium,2026-06-03 01:53:25,user_42@example.com,user_42,134.06,STORE_4
8,USR_1024,2026-06-04,North,Electronics,206.35,Active,Los Angeles,24,Basic,2026-06-15 04:54:32,user_48@example.com,user_48,8.74,STORE_4
9,USR_1022,2026-06-07,South,Electronics,158.88,Active,Los Angeles,43,Standard,2026-06-08 20:16:45,user_51@example.com,user_51,247.64,STORE_3


In [30]:
# Data Transformation: Convert and rename timestamp column

# Cast 'raw_timestamp' to TimestampType and rename it to 'event_time'
from pyspark.sql.types import TimestampType
transformed_df = (transaction_df.withColumn("raw_timestamp",col("raw_timestamp").cast(TimestampType())).withColumnRenamed("raw_timestamp", "event_time"))
transformed_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)



In [31]:
# Data Aggregation: Calculate min, max, and average price

# Compute minimum, maximum, and average values for 'price'
from pyspark.sql.functions import min,max,avg
price_summary_df = transaction_df.agg(min("price").alias("min_price"),max("price").alias("max_price"),avg("price").alias("avg_price"))
price_summary_df.toPandas()

,min_price,max_price,avg_price
0,8.35,298.26,156.095


In [32]:
# Display the total number of rows in the DataFrame
print(f"Total number of records in the DataFrame: {transaction_df.count()}")

Total number of records in the DataFrame: 200


In [33]:
# Grouping Data: Count by City

# Count transactions per city
df.groupBy("city").count().show()

+-------------+-----+
|         city|count|
+-------------+-----+
|      Phoenix|    9|
|  Los Angeles|  133|
|San Francisco|    9|
|      Chicago|   12|
|      Houston|   15|
|     New York|   22|
+-------------+-----+



In [34]:
# Calculate total sales per product category
from pyspark.sql.functions import sum
transaction_df.groupBy("product_category").agg(sum("sale_amount").alias("total_sales")).show()

+----------------+------------------+
|product_category|       total_sales|
+----------------+------------------+
|            Home|           8649.94|
|     Electronics| 8787.519999999999|
|        Clothing|          10232.62|
|           Books|          13065.36|
|          Beauty|11455.320000000002|
+----------------+------------------+



In [37]:
# Calculate average sales per region
from pyspark.sql.functions import avg
transaction_df.groupBy("region").agg(avg("sale_amount").alias("avg_sales")).show()

+------+------------------+
|region|         avg_sales|
+------+------------------+
| South|283.47900000000004|
|  East| 294.4607894736842|
|  West|249.35625000000005|
| North|272.79086956521735|
+------+------------------+



In [ ]:
# Step 9: Building a Data Processing Pipeline

#This section outlines a simplified data processing pipeline, focusing on core operations:

#1.  **Load Dataset**: Ingest the raw data.
#2.  **Clean Data**: Handle duplicates and missing values.
#3.  **Apply Transformations**: Modify existing columns or create new ones.
#4.  **Perform Aggregations**: Summarize data (although not part of the final output, shown for demonstration).

#***Note on Output***
#When applying filters, transformations, or aggregations, the resulting DataFrames are new views or summarized data. They do not directly modify the original CSV file or the appearance of the `transaction_df` unless explicitly saved or overwritten. The following pipeline will focus on creating a processed DataFrame that incorporates cleaning and transformations.

In [39]:
# Step 1: Load the dataset
processed_transaction_df = spark.read.csv("data.csv",header=True,inferSchema=True)


# Display the first few rows of the loaded dataset
processed_transaction_df.limit(20).toPandas()

,user_id,transaction_date,region,product_category,sale_amount,status,city,age,subscription,raw_timestamp,email,username,price,store_id
0,USR_1007,2026-06-13,West,Home,177.20,Active,Los Angeles,26,Standard,2026-06-11 07:02:36,user_1@example.com,user_1,204.76,STORE_5
1,USR_1007,2026-06-13,West,Home,177.20,Active,Los Angeles,26,Standard,2026-06-11 07:02:36,user_1@example.com,user_1,204.76,STORE_5
2,USR_1001,2026-06-15,West,Home,412.94,None,Los Angeles,46,Premium,2026-06-12 03:12:01,user_2@example.com,user_2,NaN,STORE_4
3,USR_1047,2026-06-09,West,Clothing,480.79,Active,Los Angeles,59,Standard,2026-06-08 10:26:09,user_3@example.com,user_3,185.04,STORE_5
4,USR_1017,2026-06-15,East,Books,416.07,Completed,Los Angeles,46,Basic,2026-06-07 22:13:26,user_4@example.com,user_4,108.60,STORE_1
5,USR_1015,2026-06-01,West,Beauty,223.39,Active,Los Angeles,38,Basic,2026-06-09 19:58:30,user_5@example.com,user_5,225.74,STORE_2
6,USR_1014,2026-06-10,West,Beauty,483.08,None,Los Angeles,48,Basic,2026-06-14 23:46:03,user_6@example.com,user_6,186.74,STORE_5
7,USR_1008,2026-06-06,West,Home,406.02,Completed,Los Angeles,32,Basic,2026-06-12 04:33:13,user_7@example.com,user_7,69.09,STORE_3
8,USR_1047,2026-06-08,East,Books,246.91,Completed,Los Angeles,20,Standard,2026-06-09 10:42:30,user_8@example.com,user_8,46.30,STORE_5
9,USR_1006,2026-06-01,South,Books,368.20,None,Los Angeles,42,Standard,2026-06-09 12:20:11,None,user_9,220.35,STORE_2


In [40]:
# Initial Data Exploration: Row and Column Counts
print(f"Total number of rows in the dataset: {processed_transaction_df.count()}")
print(f"Total number of columns in the dataset: {len(processed_transaction_df.columns)}")

Total number of rows in the dataset: 200
Total number of columns in the dataset: 14


In [41]:
# Display the DataFrame schema, showing column names and data types
processed_transaction_df.printSchema()


root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)



In [42]:
# List all column names
print(processed_transaction_df.columns)

['user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'status', 'city', 'age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'store_id']


In [43]:
# Step 2: Clean Data and Step 4: Apply Transformations


from pyspark.sql.functions import col, when
from pyspark.sql.types import TimestampType

final_processed_df = (
    processed_transaction_df
    .dropDuplicates(["user_id","transaction_date"])
    .na.fill({"status":"Unknown"})
    .withColumn(
        "event_time",
        col("raw_timestamp").cast(TimestampType())
    )
    .drop("raw_timestamp")
    .withColumn(
        "age_group",
        when(col("age") < 30, "Young")
        .when(col("age") < 50, "Adult")
        .otherwise("Senior")
    )
)

In [44]:
final_processed_df.toPandas().to_csv("final_processed_data.csv", index=False)

In [45]:
# Step 5: Perform Aggregations on the Original Data

from pyspark.sql.functions import count, sum, avg, min, max

transaction_df.agg(count("*").alias("total_rows"),sum("sale_amount").alias("total_sales"),avg("sale_amount").alias("average_sales"),min("sale_amount").alias("minimum_sales"),
max("sale_amount").alias("maximum_sales")
).show()

+----------+------------------+------------------+-------------+-------------+
|total_rows|       total_sales|     average_sales|minimum_sales|maximum_sales|
+----------+------------------+------------------+-------------+-------------+
|       200|52190.759999999966|269.02453608247407|         14.6|       498.74|
+----------+------------------+------------------+-------------+-------------+



In [46]:
# Step 3: Filter Data (Age, Category, Region)
filtered_transactions = transaction_df.filter((col("age") >= 18) & (col("age") <= 30) & (col("product_category") == "Electronics") & (col("region") == "West"))
filtered_transactions.toPandas()

,user_id,transaction_date,region,product_category,sale_amount,status,city,age,subscription,raw_timestamp,email,username,price,store_id
0,USR_1046,2026-06-13,West,Electronics,208.05,Pending,Los Angeles,22,Premium,2026-06-11 21:55:34,user_97@example.com,user_97,NaN,STORE_1
1,USR_1016,2026-06-03,West,Electronics,134.45,Active,Los Angeles,23,Premium,2026-06-15 17:41:03,user_163@example.com,user_163,76.14,STORE_2
2,USR_1043,2026-06-13,West,Electronics,132.55,None,Los Angeles,24,Standard,2026-06-08 11:03:39,user_169@example.com,user_169,74.78,STORE_4
3,USR_1000,2026-06-03,West,Electronics,321.94,None,Los Angeles,27,Standard,2026-06-02 11:25:29,user_181@example.com,user_181,242.90,STORE_4
